# Final Notebook: Deep Learning + Ensemble (Milestones 3-5)

**Key Innovation**: Cross-song stem mixing — each stem from a DIFFERENT song of the same genre.
This exactly mimics the test data distribution where mashups combine stems from different sources.

| Milestone | Model | Description |
|-----------|-------|-------------|
| 3 | GenreCNN | CNN from scratch on mel spectrograms |
| 4 | GenreCRNN | CNN + Bidirectional LSTM |
| 5 | HuBERT | Phased fine-tuning of pre-trained model |
| Final | Ensemble | Weighted soft voting + systematic TTA |

In [ ]:
!pip install transformers wandb librosa xgboost -q

In [ ]:
import sys, gc
sys.path.append('../src')
import numpy as np, pandas as pd, torch
from torch.utils.data import random_split, DataLoader
from sklearn.metrics import classification_report
from transformers import HubertForSequenceClassification
from utils import *
from train import *

set_seed()
paths = get_paths()
wandb_login()

## Build Data Loaders (GPU required from here)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Mel spectrogram dataset (for CNN/CRNN)
mel_dataset = CrossSongMelDataset(paths['STEMS_DIR'], paths['NOISE_DIR'], GENRES)
mel_train_size = int(0.85 * len(mel_dataset))
mel_val_size = len(mel_dataset) - mel_train_size
mel_train_data, mel_val_data = random_split(
    mel_dataset, [mel_train_size, mel_val_size],
    generator=torch.Generator().manual_seed(SEED)
)
mel_train_loader = DataLoader(mel_train_data, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
mel_val_loader = DataLoader(mel_val_data, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# Raw waveform dataset (for HuBERT)
hubert_dataset = CrossSongHubertDataset(paths['STEMS_DIR'], paths['NOISE_DIR'], GENRES)
hub_train_size = int(0.85 * len(hubert_dataset))
hub_val_size = len(hubert_dataset) - hub_train_size
hub_train_data, hub_val_data = random_split(
    hubert_dataset, [hub_train_size, hub_val_size],
    generator=torch.Generator().manual_seed(SEED)
)
hub_train_loader = DataLoader(hub_train_data, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
hub_val_loader = DataLoader(hub_val_data, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nMel Dataset  — Train: {mel_train_size}, Val: {mel_val_size}")
print(f"HuBERT Dataset — Train: {hub_train_size}, Val: {hub_val_size}")
print(f"Duration: {DURATION}s | Mel batch: 16 | HuBERT batch: 8")

## Milestone 3: CNN (from scratch)

In [ ]:
cnn_model = GenreCNN().to(device)
print(f"GenreCNN parameters: {sum(p.numel() for p in cnn_model.parameters()):,}")
print(cnn_model)

best_cnn_f1 = train_cnn(cnn_model, mel_train_loader, mel_val_loader, device, epochs=10)
print(f"\nBest CNN Val Macro F1: {best_cnn_f1:.4f}")

In [ ]:
# CNN Evaluation
cnn_model.load_state_dict(torch.load('best_cnn.pth'))
cnn_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in mel_val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = cnn_model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
print("CNN Per-Class Classification Report:")
print(classification_report(all_labels, all_preds, target_names=GENRES))

## Milestone 4: CRNN (CNN + BiLSTM)

In [ ]:
crnn_model = GenreCRNN().to(device)
print(f"GenreCRNN parameters: {sum(p.numel() for p in crnn_model.parameters()):,}")
print(crnn_model)

best_crnn_f1 = train_crnn(crnn_model, mel_train_loader, mel_val_loader, device, epochs=10)
print(f"\nBest CRNN Val Macro F1: {best_crnn_f1:.4f}")

In [ ]:
# CRNN Evaluation
crnn_model.load_state_dict(torch.load('best_crnn.pth'))
crnn_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in mel_val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            outputs = crnn_model(inputs)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
print("CRNN Per-Class Classification Report:")
print(classification_report(all_labels, all_preds, target_names=GENRES))

# Comparison so far
print("\n" + "=" * 50)
print("MODEL COMPARISON SO FAR")
print("=" * 50)
print(f"{'Model':<12} {'Val Macro F1':>14}")
print("-" * 28)
print(f"{'CNN':<12} {best_cnn_f1:>14.4f}")
print(f"{'CRNN':<12} {best_crnn_f1:>14.4f}")

# Cleanup CRNN for HuBERT
del crnn_model
gc.collect()
torch.cuda.empty_cache()
print("\nCRNN memory cleaned up. Ready for HuBERT.")

## Milestone 5: HuBERT Fine-Tuning (Pre-trained Model)

**Strategy**: Fine-tune `superb/hubert-base-superb-ks` with phased training:
- **Phase 1 (Epochs 1-10)**: Feature encoder FROZEN — only classifier + transformer encoder train
- **Phase 2 (Epochs 11-35)**: UNFREEZE everything — full model adapts

In [ ]:
hubert_model = HubertForSequenceClassification.from_pretrained(
    "superb/hubert-base-superb-ks",
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
).to(device)

# Phase 1: Freeze feature encoder
hubert_model.freeze_feature_encoder()

total_params = sum(p.numel() for p in hubert_model.parameters())
trainable_params = sum(p.numel() for p in hubert_model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"HuBERT Model Loaded Successfully")
print(f"  Total parameters:     {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Frozen parameters:    {frozen_params:,}")
print(f"  Feature encoder: FROZEN (Phase 1)")

best_hubert_f1 = train_hubert(hubert_model, hub_train_loader, hub_val_loader, device,
                               total_epochs=35, phase2_start=11)
print(f"\nBest HuBERT Val Macro F1: {best_hubert_f1:.4f}")

In [ ]:
# HuBERT Evaluation + Full Model Comparison
hubert_model.load_state_dict(torch.load('best_hubert.pth'))
hubert_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for waveforms, labels in hub_val_loader:
        waveforms, labels = waveforms.to(device), labels.to(device)
        with torch.amp.autocast('cuda'):
            outputs = hubert_model(waveforms).logits
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
print("HuBERT Per-Class Classification Report:")
print(classification_report(all_labels, all_preds, target_names=GENRES))

print("\n" + "=" * 55)
print("FINAL MODEL COMPARISON (All Milestones)")
print("=" * 55)
print(f"{'Model':<12} {'Val Macro F1':>14}")
print("-" * 28)
print(f"{'CNN':<12} {best_cnn_f1:>14.4f}")
print(f"{'CRNN':<12} {best_crnn_f1:>14.4f}")
print(f"{'HuBERT':<12} {best_hubert_f1:>14.4f}")
print("-" * 28)
print(f"\nBest model: HuBERT with Macro F1 = {best_hubert_f1:.4f}")
print("Target: 0.80+ on Kaggle leaderboard")

## Test Inference & Final Submission

**Key fix**: Robust file path handling — builds a lookup dictionary from actual files
in the mashups directory and tries multiple naming conventions.

**Systematic TTA**: 5 overlapping 10s windows [0-10, 5-15, 10-20, 15-25, 20-30] seconds
covering the full 30s audio.

In [ ]:
from inference import run_hubert_inference, run_ensemble_inference

# HuBERT-only submission with systematic TTA
hubert_probs, hubert_ids = run_hubert_inference(hubert_model, paths, device)

In [ ]:
# Ensemble submission (CNN + CRNN + HuBERT with weighted soft voting)
cnn_model = GenreCNN().to(device)
cnn_model.load_state_dict(torch.load('best_cnn.pth'))

crnn_model = GenreCRNN().to(device)
crnn_model.load_state_dict(torch.load('best_crnn.pth'))

sub_ensemble = run_ensemble_inference(cnn_model, crnn_model, hubert_model, paths, device)

# Cleanup
del cnn_model, crnn_model
gc.collect()
torch.cuda.empty_cache()

## WandB Final Summary

In [ ]:
import wandb

wandb.init(project="22f1001611-t12026", name="final-comparison-v3")

wandb.log({
    "cnn_val_f1": best_cnn_f1,
    "crnn_val_f1": best_crnn_f1,
    "hubert_val_f1": best_hubert_f1,
    "best_model": "HuBERT",
    "best_val_f1": best_hubert_f1,
    "duration_seconds": DURATION,
    "tta_type": "systematic_overlapping",
    "tta_windows": TTA_CROPS,
    "dataset_multiplier": DATASET_MULTIPLIER,
})

comparison_table = wandb.Table(
    columns=["Model", "Val Macro F1", "Milestone"],
    data=[
        ["CNN", round(best_cnn_f1, 4), "Milestone 3"],
        ["CRNN", round(best_crnn_f1, 4), "Milestone 4"],
        ["HuBERT", round(best_hubert_f1, 4), "Milestone 5"],
    ]
)
wandb.log({"model_comparison": comparison_table})
wandb.finish()

print("=" * 60)
print("ALL DONE! FINAL SUMMARY")
print("=" * 60)
print()
print(f"Config: DURATION={DURATION}s, TTA=Systematic {TTA_CROPS} windows,")
print(f"        Dataset={DATASET_MULTIPLIER}x multiplier")
print()
print("Model Performance (Validation Macro F1):")
print(f"  Milestone 3 — CNN:      {best_cnn_f1:.4f}")
print(f"  Milestone 4 — CRNN:     {best_crnn_f1:.4f}")
print(f"  Milestone 5 — HuBERT:   {best_hubert_f1:.4f}")
print()
print("Submission Files:")
print("  1. submission.csv          (HuBERT + Systematic TTA) <- SUBMIT THIS")
print("  2. submission_ensemble.csv (Ensemble + Systematic TTA)")
print()
print("Submit BOTH to Kaggle. Pick whichever scores higher.")
print("Target: 0.80+ Macro F1")
print("=" * 60)